In [2]:
import pandas as pd

# Load & Explore Contract Data
df=pd.read_csv('../data/batch2_contracts_125rows.csv')
df.head()

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low


In [3]:
# Create duplicate columns holding expected values for comparison
df["expected_apr"] = df["apr"]
df["expected_term"] = df["term_months"]
df["expected_payment"] = df["monthly_payment"]
df["expected_penalty"] = df["penalty_clause"]

In [ ]:
# Display all column names in the dataframe
df.columns

Index(['contract_id', 'raw_text', 'apr', 'term_months', 'monthly_payment',
       'penalty_clause', 'recommended_action', 'risk_flag', 'expected_apr',
       'expected_term', 'expected_payment', 'expected_penalty'],
      dtype='object')

In [5]:
# Display penalty vs expected penalty to verify
df[['penalty_clause',"expected_penalty"]]

,penalty_clause,expected_penalty
0,NaN,NaN
1,Early termination fee $300,Early termination fee $300
2,Late fee $50,Late fee $50
3,Late fee $25,Late fee $25
4,NaN,NaN
...,...,...
120,Late fee $25,Late fee $25
121,Late fee $25,Late fee $25
122,Early termination fee $300,Early termination fee $300
123,NaN,NaN


In [6]:
# APR score: 1 if actual == expected else 0
df["apr_score"] = (df["apr"]==df["expected_apr"]).astype(int)

# Term score: 1 if actual == expected else 0
df["term_score"] = (df["term_months"]==df["expected_term"]).astype(int)

# Payment score: 1 if actual == expected else 0
df["payment_score"] = (df["monthly_payment"]==df["expected_payment"]).astype(int)

# Penalty score: 1 if actual == expected else 0
df["penalty_score"] = (df["penalty_clause"] == df["expected_penalty"]).astype(int)

# Total score is sum of all individual scores
df["total_score"] = (df["apr_score"] + df["term_score"] + df["payment_score"] + df["penalty_score"])


In [ ]:
df["quality_score"] = (df["total_score"]/3) * 100


In [ ]:
df[["apr_score" ,"term_score","payment_score","penalty_score","total_score","quality_score"]].head(20)

,apr_score,term_score,payment_score,penalty_score,total_score,quality_score
0,1,1,1,0,3,100.000000
1,1,1,1,1,4,133.333333
2,1,1,1,1,4,133.333333
3,1,1,1,1,4,133.333333
4,1,1,1,0,3,100.000000
5,1,1,1,0,3,100.000000
6,1,1,1,0,3,100.000000
7,1,1,1,1,4,133.333333
8,1,1,1,0,3,100.000000
9,1,1,1,1,4,133.333333


In [ ]:
# Import requests library for making API calls to vehicle database
import requests

In [ ]:
def get_vehicle_details(vin):
    pass

In [ ]:
#  Vehicle API Lookup

import requests

def get_vehicle_details(vin):
    # Call NHTSA API with VIN to get vehicle information
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}?format=json"
    response = requests.get(url)
    data = response.json()
    # Return the first result which contains all vehicle details
    return data["Results"][0]

In [ ]:
# Test the function with a sample VIN
veichle = get_vehicle_details("1FTEW1EF3GKE12345")
veichle

{'ABS': '',
 'ActiveSafetySysNote': '',
 'AdaptiveCruiseControl': '',
 'AdaptiveDrivingBeam': '',
 'AdaptiveHeadlights': '',
 'AdditionalErrorText': 'The Model Year decoded for this VIN may be incorrect. If you know the Model year, please enter it and decode again to get more accurate information.',
 'AirBagLocCurtain': '',
 'AirBagLocFront': '1st Row (Driver and Passenger)',
 'AirBagLocKnee': '',
 'AirBagLocSeatCushion': '',
 'AirBagLocSide': '1st and 2nd Rows',
 'AutoReverseSystem': '',
 'AutomaticPedestrianAlertingSound': '',
 'AxleConfiguration': '',
 'Axles': '',
 'BasePrice': '',
 'BatteryA': '',
 'BatteryA_to': '',
 'BatteryCells': '',
 'BatteryInfo': '',
 'BatteryKWh': '',
 'BatteryKWh_to': '',
 'BatteryModules': '',
 'BatteryPacks': '',
 'BatteryType': '',
 'BatteryV': '',
 'BatteryV_to': '',
 'BedLengthIN': '',
 'BedType': '',
 'BlindSpotIntervention': '',
 'BlindSpotMon': '',
 'BodyCabType': 'Crew/Super Crew/Crew Max',
 'BodyClass': 'Pickup',
 'BrakeSystemDesc': '',
 'BrakeS

In [ ]:
#  Extract Only Key Vehicle Info from API response
# Filter out unnecessary data and keep only make, model, year, and body type
important_veichle_info = {
    "make":veichle.get("Make"),
    "model":veichle.get("Model"),
    "year":veichle.get("ModelYear"),
    "body_type":veichle.get("BodyClass")
}
important_veichle_info

{'make': 'FORD', 'model': 'F-150', 'year': '2016', 'body_type': 'Pickup'}

In [ ]:
veichle_year = int(important_veichle_info["year"])
if veichle_year < 2015:
    risk_level = "High"
else:
    risk_level = "Normal"

In [ ]:
# Determine Risk Level by Year
# Create cleaned vehicle data dictionary with key vehicle information
cleaned_veichle_data = {
    "make":veichle.get("Make"),
    "model":veichle.get("Model"),
    "year":veichle.get("ModelYear"),
    "body_type":veichle.get("BodyClass")
}
cleaned_veichle_data

{'make': 'FORD', 'model': 'F-150', 'year': '2016', 'body_type': 'Pickup'}

In [ ]:
# Load contract data with VIN information from CSV file
contracts_df = pd.read_csv('../data/sample_car_contracts.csv')
df.head()

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag,expected_apr,expected_term,expected_payment,expected_penalty,apr_score,term_score,payment_score,penalty_score,total_score,quality_score
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low,10.49,24,959,NaN,1,1,1,0,3,100.000000
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low,3.78,24,804,Early termination fee $300,1,1,1,1,4,133.333333
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High,5.41,24,774,Late fee $50,1,1,1,1,4,133.333333
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High,5.26,48,1028,Late fee $25,1,1,1,1,4,133.333333
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low,5.97,36,1180,NaN,1,1,1,0,3,100.000000


In [ ]:
# Function to enrich contract data with vehicle information from API
# Takes a contract row and retrieves vehicle details using the VIN
def enrich_contract_with_veichle(contract_row):
    vin = contract_row["vin"]
    # Fetch vehicle details from NHTSA API
    veichle = get_vehicle_details(vin)

    # Return enriched contract with vehicle information
    return{
        "contract_id" : contract_row["contract_id"],
        "customer_name" : contract_row["customer_name"],
        "vin" : vin,
        "veichle" : {
            "make":veichle.get("Make"),
    "model":veichle.get("Model"),
    "year":veichle.get("ModelYear"),
    }
    }

In [ ]:
# Loop through all contracts and enrich each with vehicle data
# This combines contract information with vehicle details from the API
combine_records = []

for _, row in contracts_df.iterows():
    # Append enriched contract data to the list
    combine_records.append(enrich_contract_with_veichle(row))